# Caso 2: INTERMEDIO - Calidad de Datos SISMEPRE (Impuesto Predial Municipalidades)

---

## Contexto del Proyecto

### Descripción del Problema
El **Ministerio de Economía y Finanzas (MEF)**, a través del sistema **SISMEPRE**,
registra el presupuesto y ejecución del impuesto predial de las municipalidades
peruanas. Al consolidar los datos, se detectan inconsistencias en los registros
de respuestas, valores faltantes en campos clave, y formatos irregulares que
impiden analizar correctamente el cumplimiento de metas de recaudación predial
por municipalidad.

### Objetivo Analítico
Limpiar y mejorar la calidad de los datos del SISMEPRE para:
- Imputar valores faltantes en RESPUESTA_DECIMAL y RESPUESTA_ENTERO
- Corregir inconsistencias entre SEC_EJEC y SEC_EJEC_1
- Estandarizar los campos ESTADO y CLASIFICACION
- Preparar el dataset Silver listo para el JOIN con SIAF y RENAMU

### Impacto de la Mala Calidad de Datos
- **Financiero**: Registros de recaudación predial incorrectos distorsionan
el análisis de cumplimiento de metas por municipalidad
- **Operativo**: Duplicados o respuestas inválidas generan conteos incorrectos
de municipalidades que reportaron su impuesto predial
- **Estratégico**: Decisiones de política tributaria municipal basadas en datos
incompletos pueden beneficiar o perjudicar municipios incorrectamente

---

## Dimensiones de Calidad a Corregir

En este caso aplicaremos correcciones sobre:

1. **Completitud**: Imputar nulos en RESPUESTA_DECIMAL y RESPUESTA_ENTERO
con mediana por PREGUNTA_ID
2. **Exactitud**: Corregir valores negativos en respuestas
3. **Consistencia**: Corregir inconsistencias entre SEC_EJEC y SEC_EJEC_1
4. **Integridad**: Validar que ANO_APLICACION esté en rango esperado
5. **Razonabilidad**: Tratar outliers en RESPUESTA_DECIMAL con IQR
6. **Oportunidad**: Verificar cobertura temporal por ANO_APLICACION
7. **Unicidad**: Eliminar duplicados exactos y parciales por clave de negocio
8. **Validez**: Estandarizar ESTADO y CLASIFICACION según valores permitidos

---

---

# SOLUCIÓN NIVEL INTERMEDIO - SISMEPRE

## Objetivo
Recuperar datos cuando sea posible usando:
- Imputación inteligente de valores faltantes en respuestas
- Corrección de valores negativos e inconsistencias entre campos
- Detección de outliers en RESPUESTA_DECIMAL con contexto de pregunta
- Validaciones basadas en reglas del sistema SISMEPRE

### Filosofía:
"No eliminar sin antes intentar recuperar información valiosa —
un registro SISMEPRE con nulo en RESPUESTA_DECIMAL puede imputarse
con la mediana de su misma PREGUNTA_ID y ANO_APLICACION"

In [1]:
# Instalación de librerías necesarias
# !pip install pandas numpy matplotlib seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import warnings

warnings.filterwarnings('ignore')

# =========================================================
# CONFIGURACIÓN DE VISUALIZACIÓN
# =========================================================

plt.style.use('seaborn-v0_8-darkgrid')

sns.set_palette('husl')

# Mostrar números completos
pd.set_option('display.float_format', '{:,.2f}'.format)

# Mostrar más columnas
pd.set_option('display.max_columns', None)

# Mostrar más filas
pd.set_option('display.max_rows', 100)

# =========================================================
# CONFIGURACIÓN GENERAL DEL PROYECTO
# =========================================================

RANGO_ANIOS = (2010, 2026)

COLUMNAS_DUPLICADOS = [
    'SEC_EJEC',
    'ANO_APLICACION',
    'PREGUNTA_ID'
]

COLUMNAS_NUMERICAS = [
    'RESPUESTA_DECIMAL',
    'RESPUESTA_ENTERO'
]

ESTADOS_VALIDOS = ['A', 'I']

CLASIFICACION_VALIDA = ['A', 'B']

print("✅ Librerías y configuración cargadas correctamente")

✅ Librerías y configuración cargadas correctamente


In [2]:
# =========================================================
# CARGA DEL DATASET SISMEPRE
# =========================================================

print("=" * 60)
print("         CARGA DE DATOS SISMEPRE")
print("=" * 60)

# ---------------------------------------------------------
# CARGAR ARCHIVO CSV
# ---------------------------------------------------------

df = pd.read_csv(
    'SISMEPRE_CAPA_SILVER.csv',
    encoding='latin-1',
    low_memory=False
)

# =========================================================
# INFORMACIÓN GENERAL
# =========================================================

print("\n✅ Dataset SISMEPRE cargado correctamente")

print(f"\n📊 Registros : {len(df):,}")

print(f"📊 Columnas  : {len(df.columns):,}")

# ---------------------------------------------------------
# MEMORIA UTILIZADA
# ---------------------------------------------------------

memoria_mb = (
    df.memory_usage(deep=True).sum()
    / 1024**2
)

print(f"💾 Memoria utilizada : {memoria_mb:.2f} MB")

# =========================================================
# VISTA PREVIA
# =========================================================

print("\n📌 Primeros registros:")
display(df.head())

print("\n📌 Tipos de datos:")
display(df.dtypes)

         CARGA DE DATOS SISMEPRE

✅ Dataset SISMEPRE cargado correctamente

📊 Registros : 100,000
📊 Columnas  : 27
💾 Memoria utilizada : 24.49 MB

📌 Primeros registros:


,SEC_EJEC,ANO_APLICACION,USUARIO_CREACION_FECHA,ESTADO,USUARIO_ENVIO_ID,USUARIO_FECHA_ENVIO,CORREO,ORIGEN_INFORMACION,CLASIFICACION,PERIODO,TIPO_META,IND_RESOL_ALCAL_ADJUNTO,FECHA_RESOL_ALCAL_ADJUNTO,FORMULARIO_ID,ANO_ESTADISTICA,MES_ESTADISTICA,ESTADO_REGISTRO,ANO_ESTADISTICA_DESC,SEC_EJEC_1,FORMULARIO_ID_1,PREGUNTA_ID,RESPUESTA_ID,RESPUESTA_TEXTO,RESPUESTA_DECIMAL,RESPUESTA_ENTERO,RESPUESTA_FECHA,ESTADO_REGISTRO_1
0,301003,2022,2022-01-25 18:20:51,A,0,,,1,A,1,2,A,31/03/2022 16:54:51,8,2021,13,A,2021,301071,5,86,75,0,0.00,0,,A
1,301046,2022,2022-01-25 18:20:52,A,0,,,1,A,1,2,A,15/02/2022 10:20:49,8,2021,13,A,2021,301071,5,86,75,0,0.00,0,,A
2,301052,2022,2022-01-25 18:20:52,A,0,,,1,A,1,2,A,28/03/2022 08:23:21,8,2021,13,A,2021,301071,5,86,75,0,0.00,0,,A
3,301090,2022,2022-01-25 18:20:52,A,0,,,1,A,1,2,A,31/03/2022 09:00:04,8,2021,13,A,2021,301071,5,86,75,0,0.00,0,,A
4,301098,2022,2022-01-25 18:20:52,A,0,,,1,A,1,2,P,0,8,2021,13,A,2021,301071,5,86,75,0,0.00,0,,A



📌 Tipos de datos:


SEC_EJEC                       int64
ANO_APLICACION                 int64
USUARIO_CREACION_FECHA           str
ESTADO                           str
USUARIO_ENVIO_ID               int64
USUARIO_FECHA_ENVIO              str
CORREO                           str
ORIGEN_INFORMACION             int64
CLASIFICACION                    str
PERIODO                        int64
TIPO_META                      int64
IND_RESOL_ALCAL_ADJUNTO          str
FECHA_RESOL_ALCAL_ADJUNTO        str
FORMULARIO_ID                  int64
ANO_ESTADISTICA                int64
MES_ESTADISTICA                int64
ESTADO_REGISTRO                  str
ANO_ESTADISTICA_DESC           int64
SEC_EJEC_1                     int64
FORMULARIO_ID_1                int64
PREGUNTA_ID                    int64
RESPUESTA_ID                   int64
RESPUESTA_TEXTO                  str
RESPUESTA_DECIMAL            float64
RESPUESTA_ENTERO               int64
RESPUESTA_FECHA                  str
ESTADO_REGISTRO_1                str
d

In [4]:
# =========================================================
# PIPELINE SILVER INTERMEDIO — SISMEPRE
# =========================================================

# ---------------------------------------------------------
# CREAR DATAFRAME DE TRABAJO
# ---------------------------------------------------------

df_intermedio = df.copy()

print("=" * 60)
print("        PIPELINE SILVER INTERMEDIO")
print("=" * 60)

print(f"\n📊 Registros iniciales: {len(df_intermedio):,}")

# =========================================================
# 1. CONVERSIÓN DE VARIABLES NUMÉRICAS
# =========================================================

print("\n🔹 1. Conversión de variables numéricas")

for col in COLUMNAS_NUMERICAS:

    df_intermedio[col] = pd.to_numeric(
        df_intermedio[col],
        errors='coerce'
    )

print("✅ Variables numéricas convertidas")

# =========================================================
# 2. IMPUTACIÓN DE VALORES FALTANTES
# =========================================================

print("\n" + "=" * 60)
print("2. IMPUTACIÓN DE VALORES FALTANTES")
print("=" * 60)

# ---------------------------------------------------------
# RESPUESTA_DECIMAL
# ---------------------------------------------------------

nulos_decimal = (
    df_intermedio['RESPUESTA_DECIMAL']
    .isnull()
    .sum()
)

df_intermedio['RESPUESTA_DECIMAL'] = (
    df_intermedio
    .groupby('PREGUNTA_ID')['RESPUESTA_DECIMAL']
    .transform(lambda x: x.fillna(x.median()))
    .fillna(0)
)

print(
    f"✅ RESPUESTA_DECIMAL: "
    f"{nulos_decimal:,} valores imputados"
)

# ---------------------------------------------------------
# RESPUESTA_ENTERO
# ---------------------------------------------------------

nulos_entero = (
    df_intermedio['RESPUESTA_ENTERO']
    .isnull()
    .sum()
)

df_intermedio['RESPUESTA_ENTERO'] = (
    df_intermedio
    .groupby('PREGUNTA_ID')['RESPUESTA_ENTERO']
    .transform(lambda x: x.fillna(x.median()))
    .fillna(0)
)

print(
    f"✅ RESPUESTA_ENTERO: "
    f"{nulos_entero:,} valores imputados"
)

# ---------------------------------------------------------
# RESPUESTA_TEXTO
# ---------------------------------------------------------

nulos_texto = (
    df_intermedio['RESPUESTA_TEXTO']
    .isnull()
    .sum()
)

df_intermedio['RESPUESTA_TEXTO'] = (
    df_intermedio['RESPUESTA_TEXTO']
    .fillna('SIN RESPUESTA')
)

print(
    f"✅ RESPUESTA_TEXTO: "
    f"{nulos_texto:,} valores imputados"
)

# =========================================================
# 3. CORRECCIÓN DE VALORES INCORRECTOS
# =========================================================

print("\n" + "=" * 60)
print("3. CORRECCIÓN DE VALORES INCORRECTOS")
print("=" * 60)

# ---------------------------------------------------------
# RESPUESTA_DECIMAL NEGATIVA
# ---------------------------------------------------------

mask_decimal_neg = (
    df_intermedio['RESPUESTA_DECIMAL'] < 0
)

cantidad_decimal_neg = mask_decimal_neg.sum()

if cantidad_decimal_neg > 0:

    mediana_decimal = (
        df_intermedio[df_intermedio['RESPUESTA_DECIMAL'] >= 0]
        .groupby('PREGUNTA_ID')['RESPUESTA_DECIMAL']
        .transform('median')
    )

    df_intermedio.loc[
        mask_decimal_neg,
        'RESPUESTA_DECIMAL'
    ] = (
        mediana_decimal[mask_decimal_neg]
        .fillna(0)
    )

print(
    f"✅ RESPUESTA_DECIMAL negativos corregidos: "
    f"{cantidad_decimal_neg:,}"
)

# ---------------------------------------------------------
# RESPUESTA_ENTERO NEGATIVA
# ---------------------------------------------------------

mask_entero_neg = (
    df_intermedio['RESPUESTA_ENTERO'] < 0
)

cantidad_entero_neg = mask_entero_neg.sum()

if cantidad_entero_neg > 0:

    mediana_entero = (
        df_intermedio[df_intermedio['RESPUESTA_ENTERO'] >= 0]
        .groupby('PREGUNTA_ID')['RESPUESTA_ENTERO']
        .transform('median')
    )

    df_intermedio.loc[
        mask_entero_neg,
        'RESPUESTA_ENTERO'
    ] = (
        mediana_entero[mask_entero_neg]
        .fillna(0)
    )

print(
    f"✅ RESPUESTA_ENTERO negativos corregidos: "
    f"{cantidad_entero_neg:,}"
)

# =========================================================
# 4. ESTANDARIZACIÓN DE FORMATOS
# =========================================================

print("\n" + "=" * 60)
print("4. ESTANDARIZACIÓN DE FORMATOS")
print("=" * 60)

# ---------------------------------------------------------
# ESTADO
# ---------------------------------------------------------

df_intermedio['ESTADO'] = (
    df_intermedio['ESTADO']
    .astype(str)
    .str.upper()
    .str.strip()
)

# ---------------------------------------------------------
# CLASIFICACION
# ---------------------------------------------------------

df_intermedio['CLASIFICACION'] = (
    df_intermedio['CLASIFICACION']
    .astype(str)
    .str.upper()
    .str.strip()
)

# ---------------------------------------------------------
# RESPUESTA_TEXTO
# ---------------------------------------------------------

df_intermedio['RESPUESTA_TEXTO'] = (
    df_intermedio['RESPUESTA_TEXTO']
    .astype(str)
    .str.strip()
)

print("✅ Variables estandarizadas")

# =========================================================
# 5. CORRECCIÓN DE INCONSISTENCIAS
# =========================================================

print("\n" + "=" * 60)
print("5. CORRECCIÓN DE INCONSISTENCIAS")
print("=" * 60)

# ---------------------------------------------------------
# SEC_EJEC_1
# ---------------------------------------------------------

if 'SEC_EJEC_1' in df_intermedio.columns:

    inconsistencias_sec = (
        df_intermedio['SEC_EJEC']
        !=
        df_intermedio['SEC_EJEC_1']
    ).sum()

    df_intermedio['SEC_EJEC_1'] = (
        df_intermedio['SEC_EJEC']
    )

    print(
        f"✅ SEC_EJEC_1 corregido: "
        f"{inconsistencias_sec:,}"
    )

# ---------------------------------------------------------
# FORMULARIO_ID_1
# ---------------------------------------------------------

if (
    'FORMULARIO_ID' in df_intermedio.columns
    and
    'FORMULARIO_ID_1' in df_intermedio.columns
):

    inconsistencias_form = (
        df_intermedio['FORMULARIO_ID']
        !=
        df_intermedio['FORMULARIO_ID_1']
    ).sum()

    df_intermedio['FORMULARIO_ID_1'] = (
        df_intermedio['FORMULARIO_ID']
    )

    print(
        f"✅ FORMULARIO_ID_1 corregido: "
        f"{inconsistencias_form:,}"
    )

# ---------------------------------------------------------
# ESTADO_REGISTRO_1
# ---------------------------------------------------------

if (
    'ESTADO_REGISTRO' in df_intermedio.columns
    and
    'ESTADO_REGISTRO_1' in df_intermedio.columns
):

    inconsistencias_estado = (
        df_intermedio['ESTADO_REGISTRO']
        !=
        df_intermedio['ESTADO_REGISTRO_1']
    ).sum()

    df_intermedio['ESTADO_REGISTRO_1'] = (
        df_intermedio['ESTADO_REGISTRO']
    )

    print(
        f"✅ ESTADO_REGISTRO_1 corregido: "
        f"{inconsistencias_estado:,}"
    )

# =========================================================
# 6. VALIDACIÓN DE CATEGORÍAS
# =========================================================

print("\n" + "=" * 60)
print("6. VALIDACIÓN DE CATEGORÍAS")
print("=" * 60)

# ---------------------------------------------------------
# ESTADO
# ---------------------------------------------------------

mask_estado = (
    ~df_intermedio['ESTADO']
    .isin(ESTADOS_VALIDOS)
)

cantidad_estado = mask_estado.sum()

if cantidad_estado > 0:

    df_intermedio.loc[
        mask_estado,
        'ESTADO'
    ] = 'I'

print(
    f"✅ ESTADO inválidos corregidos: "
    f"{cantidad_estado:,}"
)

# ---------------------------------------------------------
# CLASIFICACION
# ---------------------------------------------------------

mask_clasificacion = (
    ~df_intermedio['CLASIFICACION']
    .isin(CLASIFICACION_VALIDA)
)

cantidad_clasificacion = (
    mask_clasificacion.sum()
)

if cantidad_clasificacion > 0:

    df_intermedio.loc[
        mask_clasificacion,
        'CLASIFICACION'
    ] = 'A'

print(
    f"✅ CLASIFICACION inválidas corregidas: "
    f"{cantidad_clasificacion:,}"
)

# =========================================================
# 7. MANEJO DE DUPLICADOS
# =========================================================

print("\n" + "=" * 60)
print("7. MANEJO DE DUPLICADOS")
print("=" * 60)

antes_dup = len(df_intermedio)

df_intermedio = df_intermedio.drop_duplicates(
    subset=COLUMNAS_DUPLICADOS,
    keep='first'
)

eliminados_dup = (
    antes_dup - len(df_intermedio)
)

print(
    f"✅ Duplicados eliminados: "
    f"{eliminados_dup:,}"
)

# =========================================================
# 8. DETECCIÓN DE OUTLIERS
# =========================================================

print("\n" + "=" * 60)
print("8. DETECCIÓN DE OUTLIERS")
print("=" * 60)

Q1 = (
    df_intermedio['RESPUESTA_DECIMAL']
    .quantile(0.25)
)

Q3 = (
    df_intermedio['RESPUESTA_DECIMAL']
    .quantile(0.75)
)

IQR = Q3 - Q1

limite_superior = (
    Q3 + (3 * IQR)
)

outliers = (
    df_intermedio['RESPUESTA_DECIMAL']
    > limite_superior
)

cantidad_outliers = outliers.sum()

# Solo crear bandera (NO eliminar)
df_intermedio['OUTLIER_RESPUESTA_DECIMAL'] = np.where(
    outliers,
    1,
    0
)

print(
    f"✅ Outliers detectados: "
    f"{cantidad_outliers:,}"
)

# =========================================================
# 9. VALIDACIÓN TEMPORAL
# =========================================================

print("\n" + "=" * 60)
print("9. VALIDACIÓN TEMPORAL")
print("=" * 60)

antes_anio = len(df_intermedio)

df_intermedio = df_intermedio[
    (
        df_intermedio['ANO_APLICACION']
        >=
        RANGO_ANIOS[0]
    )
    &
    (
        df_intermedio['ANO_APLICACION']
        <=
        RANGO_ANIOS[1]
    )
]

eliminados_anio = (
    antes_anio - len(df_intermedio)
)

print(
    f"✅ Registros fuera de rango eliminados: "
    f"{eliminados_anio:,}"
)

# =========================================================
# RESUMEN FINAL
# =========================================================

perdida_total = (
    len(df) - len(df_intermedio)
)

porcentaje_conservado = (
    len(df_intermedio) / len(df) * 100
)

print("\n" + "=" * 60)
print("               RESUMEN FINAL")
print("=" * 60)

print(f"\n📊 Registros iniciales : {len(df):,}")

print(f"📊 Registros finales   : {len(df_intermedio):,}")

print(
    f"📉 Pérdida total       : "
    f"{perdida_total:,} registros "
    f"({perdida_total / len(df) * 100:.2f}%)"
)

print(
    f"📈 Registros conservados : "
    f"{porcentaje_conservado:.2f}%"
)

# Liberar memoria
gc.collect()

print("\n✅ Pipeline Silver intermedio completado")

        PIPELINE SILVER INTERMEDIO

📊 Registros iniciales: 100,000

🔹 1. Conversión de variables numéricas
✅ Variables numéricas convertidas

2. IMPUTACIÓN DE VALORES FALTANTES
✅ RESPUESTA_DECIMAL: 0 valores imputados
✅ RESPUESTA_ENTERO: 0 valores imputados
✅ RESPUESTA_TEXTO: 0 valores imputados

3. CORRECCIÓN DE VALORES INCORRECTOS
✅ RESPUESTA_DECIMAL negativos corregidos: 0
✅ RESPUESTA_ENTERO negativos corregidos: 0

4. ESTANDARIZACIÓN DE FORMATOS
✅ Variables estandarizadas

5. CORRECCIÓN DE INCONSISTENCIAS
✅ SEC_EJEC_1 corregido: 99,915
✅ FORMULARIO_ID_1 corregido: 100,000
✅ ESTADO_REGISTRO_1 corregido: 0

6. VALIDACIÓN DE CATEGORÍAS
✅ ESTADO inválidos corregidos: 0
✅ CLASIFICACION inválidas corregidas: 58,515

7. MANEJO DE DUPLICADOS
✅ Duplicados eliminados: 68,125

8. DETECCIÓN DE OUTLIERS
✅ Outliers detectados: 0

9. VALIDACIÓN TEMPORAL
✅ Registros fuera de rango eliminados: 0

               RESUMEN FINAL

📊 Registros iniciales : 100,000
📊 Registros finales   : 31,875
📉 Pérdida 

In [5]:
# =========================================================
# VERIFICACIÓN POST-LIMPIEZA
# =========================================================

print("\n" + "=" * 60)
print("        VERIFICACIÓN POST-LIMPIEZA")
print("              SISMEPRE")
print("=" * 60)

# =========================================================
# MÉTRICAS GENERALES
# =========================================================

total_registros = len(df_intermedio)

total_nulos = (
    df_intermedio
    .isnull()
    .sum()
    .sum()
)

duplicados_negocio = (
    df_intermedio
    .duplicated(subset=COLUMNAS_DUPLICADOS)
    .sum()
)

# =========================================================
# VALIDACIÓN DE RESPUESTAS NEGATIVAS
# =========================================================

decimal_negativos = (
    (df_intermedio['RESPUESTA_DECIMAL'] < 0)
    .sum()
)

entero_negativos = (
    (df_intermedio['RESPUESTA_ENTERO'] < 0)
    .sum()
)

# =========================================================
# VALIDACIÓN TEMPORAL
# =========================================================

anios_fuera_rango = (
    (
        df_intermedio['ANO_APLICACION']
        < RANGO_ANIOS[0]
    )
    |
    (
        df_intermedio['ANO_APLICACION']
        > RANGO_ANIOS[1]
    )
).sum()

anio_minimo = (
    df_intermedio['ANO_APLICACION']
    .min()
)

anio_maximo = (
    df_intermedio['ANO_APLICACION']
    .max()
)

# =========================================================
# VALIDACIÓN DE CATEGORÍAS
# =========================================================

estados_unicos = sorted(
    df_intermedio['ESTADO']
    .astype(str)
    .unique()
)

clasificaciones_unicas = sorted(
    df_intermedio['CLASIFICACION']
    .astype(str)
    .unique()
)

# =========================================================
# VALIDACIÓN DE OUTLIERS
# =========================================================

if 'OUTLIER_RESPUESTA_DECIMAL' in df_intermedio.columns:

    total_outliers = (
        df_intermedio['OUTLIER_RESPUESTA_DECIMAL']
        .sum()
    )

else:

    total_outliers = 0

# =========================================================
# ENTIDADES ÚNICAS
# =========================================================

total_preguntas = (
    df_intermedio['PREGUNTA_ID']
    .nunique()
)

total_sec_ejec = (
    df_intermedio['SEC_EJEC']
    .nunique()
)

# =========================================================
# REPORTE FINAL
# =========================================================

print(f"\n📊 Total registros                 : {total_registros:,}")

print(f"✅ Valores nulos totales           : {total_nulos:,}")

# ---------------------------------------------------------
# COLUMNAS CON NULOS
# ---------------------------------------------------------

nulos_por_columna = (
    df_intermedio
    .isnull()
    .sum()
)

nulos_por_columna = (
    nulos_por_columna[nulos_por_columna > 0]
    .sort_values(ascending=False)
)

if len(nulos_por_columna) > 0:

    print("\n📌 Columnas con valores nulos:")
    print(nulos_por_columna)

else:

    print("\n✅ No existen valores nulos")

# ---------------------------------------------------------
# DUPLICADOS
# ---------------------------------------------------------

print(
    f"\n✅ Duplicados clave negocio        : "
    f"{duplicados_negocio:,}"
)

# ---------------------------------------------------------
# RESPUESTAS NEGATIVAS
# ---------------------------------------------------------

print(
    f"✅ RESPUESTA_DECIMAL negativas     : "
    f"{decimal_negativos:,}"
)

print(
    f"✅ RESPUESTA_ENTERO negativas      : "
    f"{entero_negativos:,}"
)

# ---------------------------------------------------------
# VALIDACIÓN TEMPORAL
# ---------------------------------------------------------

print(
    f"✅ Registros fuera de rango        : "
    f"{anios_fuera_rango:,}"
)

print("\n" + "-" * 60)

print(
    f"📅 Cobertura temporal              : "
    f"{anio_minimo} - {anio_maximo}"
)

# ---------------------------------------------------------
# ENTIDADES
# ---------------------------------------------------------

print(
    f"📝 Preguntas distintas             : "
    f"{total_preguntas:,}"
)

print(
    f"🏢 SEC_EJEC distintos              : "
    f"{total_sec_ejec:,}"
)

# ---------------------------------------------------------
# OUTLIERS
# ---------------------------------------------------------

print(
    f"⚠️ Outliers detectados             : "
    f"{int(total_outliers):,}"
)

print("\n" + "-" * 60)

# ---------------------------------------------------------
# CATEGORÍAS
# ---------------------------------------------------------

print("📌 Valores únicos ESTADO:")
print(f"   {estados_unicos}")

print("\n📌 Valores únicos CLASIFICACION:")
print(f"   {clasificaciones_unicas}")

# ---------------------------------------------------------
# VALIDACIÓN DE CATEGORÍAS
# ---------------------------------------------------------

estados_invalidos = [
    x for x in estados_unicos
    if x not in ESTADOS_VALIDOS
]

clasificaciones_invalidas = [
    x for x in clasificaciones_unicas
    if x not in CLASIFICACION_VALIDA
]

if len(estados_invalidos) > 0:

    print("\n⚠️ ESTADOS inválidos detectados:")
    print(estados_invalidos)

else:

    print("\n✅ Todos los ESTADOS son válidos")

if len(clasificaciones_invalidas) > 0:

    print("\n⚠️ CLASIFICACIONES inválidas detectadas:")
    print(clasificaciones_invalidas)

else:

    print("✅ Todas las CLASIFICACIONES son válidas")

print("\n✅ Verificación completada correctamente")


        VERIFICACIÓN POST-LIMPIEZA
              SISMEPRE

📊 Total registros                 : 31,875
✅ Valores nulos totales           : 0

✅ No existen valores nulos

✅ Duplicados clave negocio        : 0
✅ RESPUESTA_DECIMAL negativas     : 0
✅ RESPUESTA_ENTERO negativas      : 0
✅ Registros fuera de rango        : 0

------------------------------------------------------------
📅 Cobertura temporal              : 2022 - 2022
📝 Preguntas distintas             : 75
🏢 SEC_EJEC distintos              : 425
⚠️ Outliers detectados             : 0

------------------------------------------------------------
📌 Valores únicos ESTADO:
   ['A']

📌 Valores únicos CLASIFICACION:
   ['A', 'B']

✅ Todos los ESTADOS son válidos
✅ Todas las CLASIFICACIONES son válidas

✅ Verificación completada correctamente


In [7]:
# =========================================================
# EXPORTACIÓN FINAL DATASET SILVER INTERMEDIO
# =========================================================

nombre_archivo = "sismepre_silver_intermedio.parquet"

df_intermedio.to_parquet(
    nombre_archivo,
    index=False
)

print("\n" + "=" * 60)
print("         EXPORTACIÓN COMPLETADA")
print("=" * 60)

print(f"\n✅ Archivo generado correctamente")

print(f"\n📁 Nombre archivo   : {nombre_archivo}")

print(f"📊 Registros        : {len(df_intermedio):,}")

print(f"📋 Columnas         : {len(df_intermedio.columns):,}")

print(
    f"💾 Tamaño memoria   : "
    f"{df_intermedio.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

print("\n✅ Dataset Silver Intermedio listo para análisis")


         EXPORTACIÓN COMPLETADA

✅ Archivo generado correctamente

📁 Nombre archivo   : sismepre_silver_intermedio.parquet
📊 Registros        : 31,875
📋 Columnas         : 28
💾 Tamaño memoria   : 8.08 MB

✅ Dataset Silver Intermedio listo para análisis


### Conclusiones de la Solución Intermedia

**Ventajas:**
- Elimina completamente los duplicados de negocio
- Estandariza formatos y categorías del sistema
- Aplica reglas de negocio
- Trata inconsistencias de formato

**Desventajas:**
- La eliminación de duplicados genera alta pérdida de registros (~68%)
- La imputación puede introducir sesgos
- Requiere definir previamente reglas de validación del dominio